In [ ]:
ReportFolderName = 'GPT-Based_Proposed_Version'
model_alias = "Gemma3"
random_state = 43

In [ ]:
import shutil
import os

# Keywords for folders to delete
folders_to_delete = ["logs", ReportFolderName, "results", "sample_data"]

# Delete matching folders
for item in os.listdir("."):
    if os.path.isdir(item) and any(keyword in item for keyword in folders_to_delete):
        shutil.rmtree(item)
        print(f"✅ Deleted folder: {item}")

# Delete all files in the current directory
for item in os.listdir("."):
    if os.path.isfile(item):
        os.remove(item)
        print(f"🗑️ Deleted file: {item}")

print("\n🎯 Full cleanup completed. All matching folders and all files removed.")

✅ Deleted folder: sample_data

🎯 Full cleanup completed. All matching folders and all files removed.


In [ ]:
reports_dir = f"{ReportFolderName}"
os.makedirs(reports_dir, exist_ok=True)

In [ ]:
!pip install -q -U transformers datasets peft accelerate bitsandbytes trl
!pip install -q scikit-learn pandas numpy tqdm "torchao>=0.16.0"

import os
os.environ["WANDB_MODE"] = "disabled"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 142.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 108.6 MB/s eta 0:00:00


In [ ]:

# ============================================
# STEP 1: INSTALL DEPENDENCIES
# ============================================
# Uncomment these lines if running in a new Colab environment
# !pip install -q -U transformers datasets peft accelerate bitsandbytes trl
# !pip install -q scikit-learn pandas numpy tqdm

import torch
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from tqdm import tqdm

# ============================================
# STEP 2: CONFIGURATION
# ============================================
MODEL_ID = "google/gemma-3-4b-it"
VALID_LABELS = ["smish", "promo", "normal"]

In [ ]:
# ============================================
# STEP 3: LOAD DATA & PREPARE SPLITS
# ============================================
print("\n📥 Loading SMS dataset...")
dataset = load_dataset("shariul-islam/bengali-sms-smishing-dataset")

# Shuffle and select samples
all_data = dataset['train'].shuffle(seed=42)

train_samples = dataset['train']
test_samples = dataset['test']
validation_samples = dataset['validation']

# Select Train (100) and Test (20) samples
# train_samples = all_data.select(range(100))
# test_samples = all_data.select(range(100, 120))
# validation_samples = all_data.select(range(100, 110))


print(f"✅ Train samples: {len(train_samples)}")
print(f"✅ Test samples: {len(test_samples)}")
print(f"✅ validation samples: {len(validation_samples)}")



📥 Loading SMS dataset...


README.md:   0%|          | 0.00/3.03k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/404k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/60.6k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/118k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4903 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/701 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1401 [00:00<?, ? examples/s]

✅ Train samples: 4903
✅ Test samples: 1401
✅ validation samples: 701


In [ ]:

# ============================================
# STEP 4: PROMPT TEMPLATES & HELPERS
# ============================================
def get_zero_shot_prompt(sms: str) -> str:
    return f'''You are an expert in SMS content classification for fraud detection and marketing analysis.

SMS: "{sms}"

Task: Classify the above SMS message into one of three categories:
1. smish — Fraudulent or scam SMS that tries to trick users into revealing sensitive information, clicking malicious links, or calling scam numbers.
2. promo — Promotional or marketing SMS offering discounts, sales, cashback, or advertisements.
3. normal — Regular personal messages, greetings, casual conversations.

Instructions:
- Response will only be either smish, promo, or normal.
- A single-word response.

Response:'''

def get_training_prompt(sms: str, label: str) -> str:
    return f"{get_zero_shot_prompt(sms)} {label}"

def prepare_training_data(samples) -> Dataset:
    texts = []
    for sample in samples:
        prompt = get_training_prompt(sample['text'], sample['label'])
        texts.append({"text": prompt})
    return Dataset.from_list(texts)

def classify_sms(model, tokenizer, sms_text: str) -> str:
    """Classify a single SMS message"""
    prompt = get_zero_shot_prompt(sms_text)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.2,
        )

    generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    response = generated.strip().lower()

    # Simple mapping to handle extra punctuation
    if 'smish' in response: return 'smish'
    if 'promo' in response: return 'promo'
    if 'normal' in response: return 'normal'
    return "unknown"

def classify_sms_with_probs(model, tokenizer, sms_text: str) -> tuple:
    """Classify SMS with probability scores for each class"""
    prompt = get_zero_shot_prompt(sms_text)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Get the token IDs for our labels
    # We check the first token of each label
    smish_token = tokenizer.encode("smish", add_special_tokens=False)[0]
    promo_token = tokenizer.encode("promo", add_special_tokens=False)[0]
    normal_token = tokenizer.encode("normal", add_special_tokens=False)[0]

    with torch.no_grad():
        # Get logits instead of generating
        outputs = model(**inputs)

        # Get logits for the next token (last position)
        next_token_logits = outputs.logits[0, -1, :]

        # Apply softmax to get probabilities
        probs = torch.softmax(next_token_logits, dim=-1)

        # Extract probabilities for our target labels
        prob_smish = probs[smish_token].item()
        prob_promo = probs[promo_token].item()
        prob_normal = probs[normal_token].item()

        # Normalize to sum to 1 (since we only care about these 3)
        total = prob_smish + prob_promo + prob_normal
        prob_smish_norm = prob_smish / total
        prob_promo_norm = prob_promo / total
        prob_normal_norm = prob_normal / total

        # Get predicted class
        probs_dict = {
            'smish': prob_smish_norm,
            'promo': prob_promo_norm,
            'normal': prob_normal_norm
        }
        predicted = max(probs_dict, key=probs_dict.get)

    return predicted, probs_dict



In [ ]:
# ============================================
# STEP 5: LOAD BASE MODEL (QUANTIZED)
# ============================================
print("\n⚙️ Loading Base Model (4-bit)...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16, # Explicitly set this
    trust_remote_code=True,
)


⚙️ Loading Base Model (4-bit)...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

In [ ]:
# ============================================
# STEP 6: EVALUATE ZERO-SHOT (BASELINE)
# ============================================
print("\n" + "="*50)
print("🔍 EVALUATION 1: ZERO-SHOT BASELINE")
print("="*50)
print("Running inference on test_samples BEFORE training...")

y_true = [sample['label'] for sample in test_samples]
y_pred_zero = []
analysis_results = []

for sample in tqdm(test_samples, desc="Zero-Shot Inference"):
    pred, probs = classify_sms_with_probs(base_model, tokenizer, sample['text'])
    y_pred_zero.append(pred)

    # Store all relevant info in a dictionary
    analysis_results.append({
        "SMS_Text": sample['text'],
        "True_Label": sample['label'],
        "Predicted_Label": pred,
        "Prob_Smish": probs.get('smish', 0),
        "Prob_Promo": probs.get('promo', 0),
        "Prob_Normal": probs.get('normal', 0),
        "Source": sample['source'],
        "Is_Correct": 1 if pred == sample['label'] else 0
    })
    print({
            "SMS_Text": sample['text'],
            "True_Label": sample['label'],
            "Predicted_Label": pred,
            "Prob_Smish": probs.get('smish', 0),
            "Prob_Promo": probs.get('promo', 0),
            "Prob_Normal": probs.get('normal', 0),
            "Source": sample['source'],
            "Is_Correct": 1 if pred == sample['label'] else 0
        })

# Create DataFrame
df_analysis = pd.DataFrame(analysis_results)

# Save to CSV
csv_filename = "smish_detection_base_line_results.csv"
df_analysis.to_csv(f"{reports_dir}/{csv_filename}", index=False, encoding='utf-8-sig')

# Store baseline metrics
acc_zero = accuracy_score(y_true, y_pred_zero)
prec_zero, rec_zero, f1_zero, _ = precision_recall_fscore_support(y_true, y_pred_zero, average='weighted', zero_division=0)

print(f"\nBaseline Accuracy: {acc_zero:.4f}")
print("Baseline Classification Report:")
print(classification_report(y_true, y_pred_zero, zero_division=0))


🔍 EVALUATION 1: ZERO-SHOT BASELINE
Running inference on test_samples BEFORE training...


Zero-Shot Inference:   0%|          | 2/1401 [00:01<10:52,  2.14it/s]

{'SMS_Text': 'প্রিয় ব্যবহারকারী, বাংলাদেশ জাতীয় লটারি থেকে অভিনন্দন! আপনি ৫,০০,০০০ টাকা জিতেছেন। পুরস্কার দাবি করতে দ্রুত আপনার জাতীয় পরিচয়পত্রের স্ক্যান কপি এবং বিকাশ নাম্বার পাঠান bdlottery@prizeclaim.org এ।', 'True_Label': 'smish', 'Predicted_Label': 'smish', 'Prob_Smish': 0.9999990546321426, 'Prob_Promo': 6.595589702996037e-08, 'Prob_Normal': 8.794119603994716e-07, 'Source': 'Bengali', 'Is_Correct': 1}
{'SMS_Text': 'ক্যাশব্যাকে সেঞ্চুরি ৳১০০! ৩০জিবি+৫০০মি. @৳৩৯৯,৩০দিন: cutt.ly/AwkY4L2C', 'True_Label': 'promo', 'Predicted_Label': 'smish', 'Prob_Smish': 0.9740234522528403, 'Prob_Promo': 0.025958146657712327, 'Prob_Normal': 1.8401089447438445e-05, 'Source': 'Bengali', 'Is_Correct': 0}


Zero-Shot Inference:   0%|          | 4/1401 [00:01<05:49,  4.00it/s]

{'SMS_Text': 'Robi 4G blast offer: Recharge 109TK and paben 5GB high-speed 4G data valid 10 days. Amar Robi app theke activate korun today.', 'True_Label': 'promo', 'Predicted_Label': 'promo', 'Prob_Smish': 0.007106647431732502, 'Prob_Promo': 0.9923464050128294, 'Prob_Normal': 0.0005469475554380903, 'Source': 'Banglish', 'Is_Correct': 1}
{'SMS_Text': 'Pickaboo.com e smartphone flash sale, 5000TK discount selected models e.', 'True_Label': 'promo', 'Predicted_Label': 'promo', 'Prob_Smish': 0.00974181438233256, 'Prob_Promo': 0.9900424827552445, 'Prob_Normal': 0.00021570286242287857, 'Source': 'Banglish', 'Is_Correct': 1}


...

{'SMS_Text': 'শুভ রথযাত্রা!', 'True_Label': 'normal', 'Predicted_Label': 'normal', 'Prob_Smish': 0.0026285778698206404, 'Prob_Promo': 0.00027700797806960216, 'Prob_Normal': 0.9970944141521098, 'Source': 'Bengali', 'Is_Correct': 1}
{'SMS_Text': 'Received amount 2.221 Bitcoin BTC ($18,421 USD) please confirm transaction: http://bit.do/Coinbase432194-53242', 'True_Label': 'smish', 'Predicted_Label': 'smish', 'Prob_Smish': 0.9999989612095237, 'Prob_Promo': 3.454356506581043e-08, 'Prob_Normal': 1.0042469111942798e-06, 'Source': 'English', 'Is_Correct': 1}


Zero-Shot Inference: 100%|█████████▉| 1400/1401 [03:21<00:00,  6.90it/s]

{'SMS_Text': 'bKash account e trutti dekha diyeche. Druto shomadhan er jonno ekhane click korun: https://wa.me/8801719900112', 'True_Label': 'smish', 'Predicted_Label': 'smish', 'Prob_Smish': 0.9999993856688394, 'Prob_Promo': 5.200561428960002e-08, 'Prob_Normal': 5.623255462912002e-07, 'Source': 'Banglish', 'Is_Correct': 1}
{'SMS_Text': 'Banglalink ultranet saver: 2GB + 100 min call pack only 98TK. Validity 7 days. Dial *222*98# to activate now.', 'True_Label': 'promo', 'Predicted_Label': 'promo', 'Prob_Smish': 0.008070688095735058, 'Prob_Promo': 0.9914886708875438, 'Prob_Normal': 0.000440641016721167, 'Source': 'Banglish', 'Is_Correct': 1}


Zero-Shot Inference: 100%|██████████| 1401/1401 [03:21<00:00,  6.96it/s]


{'SMS_Text': 'Want to earn 500 to 5000 TK daily through network marketing? I will teach you how to market for free, no money needed. First see the work, see the proof, if you like it, work. If anyone wants to work, inbox ❤️\u200d🩹', 'True_Label': 'smish', 'Predicted_Label': 'smish', 'Prob_Smish': 0.9999956886159493, 'Prob_Promo': 2.847765194739621e-07, 'Prob_Normal': 4.026607531166721e-06, 'Source': 'English', 'Is_Correct': 1}

Baseline Accuracy: 0.7645
Baseline Classification Report:
              precision    recall  f1-score   support

      normal       0.97      0.56      0.71       498
       promo       0.95      0.68      0.80       342
       smish       0.64      0.99      0.78       561

    accuracy                           0.76      1401
   macro avg       0.85      0.75      0.76      1401
weighted avg       0.83      0.76      0.76      1401



In [ ]:
# ============================================
# STEP 7: PREPARE FOR FINE-TUNING
# ============================================
print("\n🔧 Preparing model for LoRA training...")
base_model = prepare_model_for_kbit_training(base_model)



def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Get the highest probability token for every position
    predictions = np.argmax(logits, axis=-1)

    last_token_preds = []
    last_token_labels = []

    # Loop through the batch to find the specific classification token
    for i in range(labels.shape[0]):
        # Identify non-padding indices
        valid_indices = np.where(labels[i] != -100)[0]
        if len(valid_indices) > 0:
            last_idx = valid_indices[-1]
            # Logits at [last_idx - 1] predict the label at [last_idx]
            last_token_preds.append(predictions[i, last_idx - 1])
            last_token_labels.append(labels[i, last_idx])

    # Calculate classification-only metrics
    acc = accuracy_score(last_token_labels, last_token_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        last_token_labels, last_token_preds, average='weighted', zero_division=0
    )

    return {
        'eval_accuracy': acc,
        'eval_f1': f1,
        'eval_precision': precision,
        'eval_recall': recall
    }


# data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# base_model.gradient_checkpointing_enable()
lora_config = LoraConfig(
    r=8,                          # Increased from 16 for higher linguistic capacity
    lora_alpha=32,                 # 2x Rank is the best practice
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,              # Added dropout to prevent memorizing specific phone numbers
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
train_dataset = prepare_training_data(train_samples)
val_dataset = prepare_training_data(validation_samples) # Process validation too


🔧 Preparing model for LoRA training...


In [ ]:
# ============================================
# STEP 8: TRAIN (FINE-TUNE)
# ============================================
print("\n🚀 Starting LoRA Fine-Tuning...")
training_args = SFTConfig(
    output_dir="./smishdetect-lora",
    num_train_epochs=10,               # 2 epochs is enough for 5.6k classification samples
    # per_device_train_batch_size=8,    # Increased for Colab Pro+ GPU (A100/L4)
    per_device_train_batch_size=32,    # Increased for Colab Pro+ GPU (A100/L4)
    # gradient_accumulation_steps=2,    # Effective Batch Size = 16
    # learning_rate=1e-4,               # Stable learning rate for Gemma
    learning_rate=3e-5,               # Stable learning rate for Gemma
    # max_length=128,               # SMS are short; 256 is plenty and saves VRAM
    weight_decay=0.01,                # Regularization to keep accuracy high on test data
    bf16=True,                        # Use bfloat16 for speed on modern GPUs

    # # Validation & Logging
    # eval_strategy="steps",            # Evaluate every X steps
    # eval_steps=100,                   # Check validation performance often
    # save_strategy="steps",
    # save_steps=100,
    # logging_steps=10,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    metric_for_best_model="eval_f1",
    greater_is_better=True,

    # Optimization
    # optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",       # Smoothly lowers learning rate for better accuracy
    # warmup_ratio=0.1,                 # 10% of training spent "warming up"

    load_best_model_at_end=True,      # Automatically keep the most accurate version
    report_to="none",
)
training_args = SFTConfig(
    output_dir="./smishdetect-lora",
    num_train_epochs=3,
    # Maximize your L4/A100 GPU throughput
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,
    learning_rate=1e-4,
    # max_length=256,              # Safe for prompt + SMS
    # packing=True,                    # Fills empty space for faster training
    bf16=True,                       # Fast 16-bit brain float

    # Strategy
    eval_strategy="epoch",
    save_strategy="epoch",
    metric_for_best_model="eval_f1",  # Tracks the REAL metric
    load_best_model_at_end=True,

    # Faster Optimizer
    optim="adamw_torch_fused",
    report_to="none",
    seed = random_state
)
training_args = SFTConfig(
    output_dir="./smishdetect-lora",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=False,
    bf16=True,
    logging_steps=10,
    optim="paged_adamw_8bit",
    max_grad_norm=0.3,
    lr_scheduler_type="cosine",
    seed=random_state,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
    processing_class=tokenizer,
    # compute_metrics=compute_metrics
)

trainer.train()
print("\n💾 Saving adapter...")
# trainer.model.save_pretrained("./smishdetect-lora-adapter")

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.



🚀 Starting LoRA Fine-Tuning...


/tmp/ipykernel_502/3767190168.py:5: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  training_args = SFTConfig(
/tmp/ipykernel_502/3767190168.py:38: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doe

Adding EOS to train dataset:   0%|          | 0/4903 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/4903 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/701 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/701 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 1}.


Step,Training Loss
10,1.565230
20,0.753721
30,0.634100
40,0.656195
50,0.583603
60,0.612356
70,0.533290
80,0.549073
90,0.545466
100,0.558254



💾 Saving adapter...


In [ ]:
trainer.model.save_pretrained("./gemma3-4v-fine-tune-adapter")
trainer.model.push_to_hub(
    repo_id="shariul-islam/gemma3-4v-fine-tune-adapter",
    private=False  # set True if you want private
)


README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   2%|1         |  638kB / 32.9MB            

CommitInfo(commit_url='https://huggingface.co/shariul-islam/gemma3-4v-fine-tune-adapter/commit/27f3895e0411c4b036a81c69df1966db96f5d0aa', commit_message='Upload model', commit_description='', oid='27f3895e0411c4b036a81c69df1966db96f5d0aa', pr_url=None, repo_url=RepoUrl('https://huggingface.co/shariul-islam/gemma3-4v-fine-tune-adapter', endpoint='https://huggingface.co', repo_type='model', repo_id='shariul-islam/gemma3-4v-fine-tune-adapter'), pr_revision=None, pr_num=None)

In [ ]:
# Assuming you have already loaded your tokenizer
# tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b")

# Encode the prompt to token IDs
prompt = get_training_prompt('How are you kjn klkl kll lml,m lml lmlmm', 'normal')
print(prompt)
tokens = tokenizer.encode(prompt, add_special_tokens=True)

# Count the tokens
token_count = len(tokens)

print(f"The prompt contains {token_count} tokens.")

You are an expert in SMS content classification for fraud detection and marketing analysis.

SMS: "How are you kjn klkl kll lml,m lml lmlmm"

Task: Classify the above SMS message into one of three categories:
1. smish — Fraudulent or scam SMS that tries to trick users into revealing sensitive information, clicking malicious links, or calling scam numbers.
2. promo — Promotional or marketing SMS offering discounts, sales, cashback, or advertisements.
3. normal — Regular personal messages, greetings, casual conversations.

Instructions:
- Response will only be either smish, promo, or normal.
- A single-word response.

Response: normal
The prompt contains 147 tokens.


In [ ]:
# ============================================
# STEP 9: EVALUATE FINE-TUNED MODEL
# ============================================
print("\n" + "="*50)
print("🔍 EVALUATION 2: FINE-TUNED MODEL")
print("="*50)
print("Running inference on test_samples AFTER training...")

# Note: 'model' is now the Fine-Tuned model (Base + LoRA)
y_pred = []
analysis_results = []
for sample in tqdm(test_samples, desc="Fine-Tuned Inference"):
    pred, probs = classify_sms_with_probs(model, tokenizer, sample['text'])
    y_pred.append(pred)

    # Store all relevant info in a dictionary
    analysis_results.append({
        "SMS_Text": sample['text'],
        "True_Label": sample['label'],
        "Predicted_Label": pred,
        "Prob_Smish": probs.get('smish', 0),
        "Prob_Promo": probs.get('promo', 0),
        "Prob_Normal": probs.get('normal', 0),
        "Source": sample['source'],
        "Is_Correct": 1 if pred == sample['label'] else 0
    })


# Create DataFrame
df_analysis = pd.DataFrame(analysis_results)

# Save to CSV
csv_filename = "smish_detection_fine_tuned_results.csv"
df_analysis.to_csv(f"{reports_dir}/{csv_filename}", index=False, encoding='utf-8-sig')

# Store fine-tuned metrics
acc_ft = accuracy_score(y_true, y_pred)
prec_ft, rec_ft, f1_ft, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)

print(f"\nFine-Tuned Accuracy: {acc_ft:.4f}")
print("Fine-Tuned Classification Report:")
print(classification_report(y_true, y_pred, zero_division=0))



🔍 EVALUATION 2: FINE-TUNED MODEL
Running inference on test_samples AFTER training...


Fine-Tuned Inference: 100%|██████████| 1401/1401 [05:11<00:00,  4.50it/s]


Fine-Tuned Accuracy: 0.9979
Fine-Tuned Classification Report:
              precision    recall  f1-score   support

      normal       1.00      1.00      1.00       498
       promo       1.00      1.00      1.00       342
       smish       1.00      1.00      1.00       561

    accuracy                           1.00      1401
   macro avg       1.00      1.00      1.00      1401
weighted avg       1.00      1.00      1.00      1401



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve,
    auc, precision_recall_fscore_support
)
from sklearn.preprocessing import label_binarize
import os

# --- Preparation for Metrics ---
labels = ['normal', 'promo', 'smish']
n_classes = len(labels)

# 1. GENERATE CLASSIFICATION REPORTS
report_dict = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
report_df =  pd.DataFrame(report_dict).transpose().round(4)


# Text file
report_text = classification_report(y_true, y_pred)
with open(f"{reports_dir}/classification_report.txt", "w") as f:
        f.write(report_text)

# Save as CSV and LaTeX
report_df.to_csv(f"{reports_dir}/classification_report.csv")
report_df.to_latex(f"{reports_dir}/classification_report.tex", float_format="%.4f")

# 2. CONFUSION MATRIX (Overall)
cm = confusion_matrix(y_true, y_pred, labels=labels)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title(f'Confusion Matrix - {model_alias}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.savefig(f"{reports_dir}/{model_alias}_confusion_matrix.png")
plt.close()

# 3. CONFUSION MATRIX (Per-Source)
all_source_reports = []  # store metrics for summary
for source in df_analysis['Source'].unique():
    source_df = df_analysis[df_analysis['Source'] == source]
    y_true_src = source_df['True_Label']
    y_pred_src = source_df['Predicted_Label']
    cm_s = confusion_matrix(y_true_src, y_pred_src, labels=labels)

    plt.figure(figsize=(6, 4))
    sns.heatmap(cm_s, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.title(f'Confusion Matrix - {model_alias} ({source})')
    plt.savefig(f"{reports_dir}/{model_alias}_confusion_matrix_{source}.png")
    plt.close()

    # --- Classification Report ---
    report_dict = classification_report(
        y_true_src, y_pred_src,
        output_dict=True,
        zero_division=0
    )
    report_df = pd.DataFrame(report_dict).transpose().round(4)
    report_df.to_csv(f"{reports_dir}/{model_alias}_classification_report_{source}.csv", index=True)

    # Add macro averages for summary
    all_source_reports.append({
        "source": source,
        "precision": round(report_dict["macro avg"]["precision"], 4),
        "recall": round(report_dict["macro avg"]["recall"], 4),
        "f1_score": round(report_dict["macro avg"]["f1-score"], 4)
        })
# --- Summary Report Across Sources ---
summary_df = pd.DataFrame(all_source_reports)
summary_df.loc["Average"] = summary_df.mean(numeric_only=True)
summary_df.to_csv(f"{reports_dir}/{model_alias}_source_summary_report.csv", index=False)

print("\n✅ Per-source classification reports saved.")
print(f"✅ Summary report saved to: {reports_dir}/{model_alias}_source_summary_report.csv")

# 4. ROC CURVE (One-vs-Rest)
# Convert y_true and probabilities to binarized format for multi-class ROC
y_true_bin = label_binarize(y_true, classes=labels)
y_score = df_analysis[['Prob_Normal', 'Prob_Promo', 'Prob_Smish']].values

plt.figure(figsize=(10, 8))
for i in range(n_classes):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_score[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'ROC {labels[i]} (AUC = {roc_auc:.2f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Multi-class ROC Curve (One-vs-Rest)')
plt.legend(loc="lower right")
plt.savefig(f"{reports_dir}/roc_curve_combined.png")
plt.close()

# 5. APPEND TO SUMMARY CSV
summary_data = {
    "Model_Name": model_alias,
    "Accuracy": round(acc_ft, 4),
    "Precision": round(prec_ft, 4),
    "Recall": round(rec_ft, 4),
    "F1": round(f1_ft, 4),
    "Timestamp": pd.Timestamp.now()
}

summary_df = pd.DataFrame([summary_data])
summary_df.to_csv(f"{reports_dir}/summary.csv", index=False)

print(f"✅ Evaluation reports generated in: {reports_dir}")


✅ Per-source classification reports saved.
✅ Summary report saved to: GPT-Based_Proposed_Version/Gemma3_source_summary_report.csv
✅ Evaluation reports generated in: GPT-Based_Proposed_Version


In [ ]:
import shutil
from google.colab import files
import os

# List of folders to zip and download
folders_to_download = [
    ReportFolderName,
    #"stacking_ensemble_reports"
]

for folder in folders_to_download:
    if os.path.exists(folder):
        zip_filename = f"{folder}.zip"
        # Create zip archive
        shutil.make_archive(folder, 'zip', folder)
        # Download zip
        files.download(zip_filename)
        print(f"✅ Download started for '{zip_filename}'")
    else:
        print(f"⚠️ Folder '{folder}' not found")



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download started for 'GPT-Based_Proposed_Version.zip'


In [ ]:
# ============================================
# STEP 10: COMPARISON RESULTS
# ============================================
print("\n" + "="*50)
print("📊 FINAL COMPARISON: ZERO-SHOT vs FINE-TUNED")
print("="*50)

results_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision (W)', 'Recall (W)', 'F1-Score (W)'],
    'Zero-Shot (Base)': [acc_zero, prec_zero, rec_zero, f1_zero],
    'Fine-Tuned (LoRA)': [acc_ft, prec_ft, rec_ft, f1_ft]
})

# Add Improvement Column
results_df['Improvement'] = results_df['Fine-Tuned (LoRA)'] - results_df['Zero-Shot (Base)']

# Formatting for display
print(results_df.round(4).to_markdown(index=False))

print("\nDetailed Confusion Matrix Comparison:")
print("\n--- Zero-Shot Confusion Matrix ---")
print(confusion_matrix(y_true, y_pred_zero, labels=VALID_LABELS))
print("\n--- Fine-Tuned Confusion Matrix ---")
print(confusion_matrix(y_true, y_pred, labels=VALID_LABELS))


📊 FINAL COMPARISON: ZERO-SHOT vs FINE-TUNED
| Metric        |   Zero-Shot (Base) |   Fine-Tuned (LoRA) |   Improvement |
|:--------------|-------------------:|--------------------:|--------------:|
| Accuracy      |             0.7645 |              0.9979 |        0.2334 |
| Precision (W) |             0.8339 |              0.9979 |        0.164  |
| Recall (W)    |             0.7645 |              0.9979 |        0.2334 |
| F1-Score (W)  |             0.7599 |              0.9979 |        0.2379 |

Detailed Confusion Matrix Comparison:

--- Zero-Shot Confusion Matrix ---
[[557   4   0]
 [ 99 233  10]
 [210   7 281]]

--- Fine-Tuned Confusion Matrix ---
[[559   0   2]
 [  1 341   0]
 [  0   0 498]]
